In [1]:
import pandas as pd
import plotly.express as px
import plotly.subplots as sp

In [2]:
layout = {"height" : 850, "width": 1000, "template" : "plotly_white",
        "title" : "",
        "font":{"size" : 28},

       'xaxis1': {
        'zerolinewidth': 2,
        'zerolinecolor': 'black',
        'showticklabels': True,
        "tickangle": -45,  
        'showline': True, 
        "title": "Date",
        'linewidth': 2,  
        'linecolor': 'black',  
        'mirror': True  # This creates the frame by mirroring the axis line
    },
    "yaxis1": {
        "side": 'left',
        "range": [0, 24],
        "dtick": 4,
        'showline': True,
        'linewidth': 2,
        'linecolor': 'black',
        'mirror': True
    },
   
        "legend" : {"tracegroupgap":8 ,"font_size": 28, "orientation":"h", "yanchor":"bottom","xanchor":"center",
                    "y":-0.152,"x":0.5, "title" : "","itemwidth" : 60
                    },
                    
                    "showlegend": False

        }

In [3]:
dir = "data/"

df_france = pd.read_csv(f"{dir}France_results.csv", sep = ",", decimal = ".", index_col = "datetime", parse_dates = True)


df_france["electric_emissions_ref"] = df_france[["Electricity_Consumption[MW]" , "Taux de Co2"]].product(axis= 1) * 1000

df_france["thermal_emissions_ref"] = df_france[["Heat_demand[MW]" , "cciag_co2"]].product(axis= 1) * 1000

df_france["Reference_emissions"] = df_france[["electric_emissions_ref", "thermal_emissions_ref"]].sum(axis = 1)

df_france["Reference_emissions"]



df_czech = pd.read_csv(f"{dir}Czech_results.csv", sep = ",", decimal = ".", index_col = "datetime", parse_dates = True)

df_czech["electric_emissions_ref"] = df_czech[["Electricity_Consumption[MW]" , "Taux de Co2"]].product(axis= 1) * 1000

df_czech["thermal_emissions_ref"] = df_czech[["Heat_demand[MW]" , "cciag_co2"]].product(axis= 1) * 1000

df_czech["Reference_emissions"] = df_czech[["electric_emissions_ref", "thermal_emissions_ref"]].sum(axis = 1)

df_czech["Reference_emissions"]

datetime
2023-01-24 00:00:00    60744.688333
2023-01-24 01:00:00    62633.985000
2023-01-24 02:00:00    67360.735000
2023-01-24 03:00:00    67352.386667
2023-01-24 04:00:00    97715.908333
                           ...     
2023-05-08 19:00:00    27244.208333
2023-05-08 20:00:00    25156.323333
2023-05-08 21:00:00    27578.826667
2023-05-08 22:00:00    25343.866667
2023-05-08 23:00:00    27626.866667
Name: Reference_emissions, Length: 2520, dtype: float64

In [ ]:
df_france[["estimated_CO2_S3_total","estimated_CO2_S4_total"]]/1e6

KeyError: "['estimated_CO2_S3_total'] not in index"

In [ ]:
df_france[["estimated_CO2_S3_total","estimated_CO2_S4_total"]]/1e6

,estimated_CO2_S3_total,estimated_CO2_S4_total
datetime,,
2023-01-24 00:00:00,0.012466,0.012466
2023-01-24 01:00:00,0.014103,0.014103
2023-01-24 02:00:00,0.015266,0.015266
2023-01-24 03:00:00,0.015497,0.015497
2023-01-24 04:00:00,0.024048,0.024048
...,...,...
2023-05-08 19:00:00,0.000000,0.000000
2023-05-08 20:00:00,0.000000,0.000000
2023-05-08 21:00:00,0.000000,0.000000


In [ ]:
cols = [col for col in df_france.columns if "S1" in col ]
cols.extend(["Electricity_Consumption[MW]", "Heat_demand[MW]"])
px.line(df_france[cols])

In [ ]:
df_france[[col for col in df_france.columns if "emis" in col or "total" in col ]].sum()/1e6

estimated_CO2_S1_total    240.177789
estimated_CO2_S2_total    212.705940
estimated_CO2_S3_total    167.471153
estimated_CO2_S4_total    167.471153
electric_emissions_ref    177.159810
thermal_emissions_ref      63.039224
Reference_emissions       240.199034
dtype: float64

In [ ]:
lca_years = 30
gwp = (2204/1350) * 1000  # gco2/kwh

gwp_storage = ( gwp /lca_years) *  (len(df_france.resample("1D").mean())/365)



In [ ]:
emissions_fr= df_france[[col for col in df_france.columns if "emis" in col or "total" in col ]].sum()
emissions_fr/1e6




estimated_CO2_S1_total    240.177789
estimated_CO2_S2_total    212.705940
estimated_CO2_S3_total    167.471153
estimated_CO2_S4_total    167.471153
electric_emissions_ref    177.159810
thermal_emissions_ref      63.039224
Reference_emissions       240.199034
dtype: float64

In [ ]:
emissions_fr['estimated_CO2_S3_total'] += (gwp_storage * df_france[[col for col in df_france.columns if "Cap" in col ]].mean()['Buffer_Cap_S3_KWp'])
emissions_fr['estimated_CO2_S4_total'] += (gwp_storage * df_france[[col for col in df_france.columns if "Cap" in col ]].mean()['Buffer_Cap_S4_KWp'])
emissions_fr/1e6

estimated_CO2_S1_total    240.177789
estimated_CO2_S2_total    212.705940
estimated_CO2_S3_total    167.960996
estimated_CO2_S4_total    167.960996
electric_emissions_ref    177.159810
thermal_emissions_ref      63.039224
Reference_emissions       240.199034
dtype: float64

In [ ]:
(1 - (emissions_fr / emissions_fr["Reference_emissions"])) * 100

estimated_CO2_S1_total     0.008845
estimated_CO2_S2_total    11.445963
estimated_CO2_S3_total    30.074242
estimated_CO2_S4_total    30.074242
electric_emissions_ref    26.244579
thermal_emissions_ref     73.755421
Reference_emissions        0.000000
dtype: float64

In [ ]:
emissions_cz= df_czech[[col for col in df_czech.columns if "emis" in col or "total" in col ]].sum()

emissions_cz/1e6

estimated_CO2_S1_total    1750.822920
estimated_CO2_S2_total    1575.800775
estimated_CO2_S3_total    1947.764958
estimated_CO2_S4_total    1354.157592
electric_emissions_ref    2002.937407
thermal_emissions_ref       63.039224
Reference_emissions       2065.976631
dtype: float64

In [ ]:
emissions_cz['estimated_CO2_S3_total'] += (gwp_storage * df_czech[[col for col in df_czech.columns if "Cap" in col ]].mean()['Buffer_Cap_S3_KWp'])
emissions_cz['estimated_CO2_S4_total'] += (gwp_storage * df_czech[[col for col in df_czech.columns if "Cap" in col ]].mean()['Buffer_Cap_S4_KWp'])

(emissions_cz/1e6).round(2)

estimated_CO2_S1_total    1750.82
estimated_CO2_S2_total    1575.80
estimated_CO2_S3_total    1947.76
estimated_CO2_S4_total    1354.53
electric_emissions_ref    2002.94
thermal_emissions_ref       63.04
Reference_emissions       2065.98
dtype: float64

In [ ]:
((1 - (emissions_cz / emissions_cz["Reference_emissions"])) * 100).round(2)

estimated_CO2_S1_total    15.25
estimated_CO2_S2_total    23.73
estimated_CO2_S3_total     5.72
estimated_CO2_S4_total    34.44
electric_emissions_ref     3.05
thermal_emissions_ref     96.95
Reference_emissions        0.00
dtype: float64

In [ ]:
(df_france[[col for col in df_czech.columns if "Cap" in col ]].mean())

PV_Cap_S1_KWp           75.456448
PV_Cap_S2_KWp            0.000000
Buffer_Cap_S3_KWp    31289.843750
PV_Cap_S4_KWp            0.000000
Buffer_Cap_S4_KWp    31289.843750
dtype: float64

In [ ]:
layout = {"height" : 1800, "width": 2000, "template" : "plotly_white",
        "font":{"size" : 22},

        "legend" : {"tracegroupgap":8 ,"font_size": 28, "orientation":"h", "yanchor":"bottom","xanchor":"center",
                    "y":-0.12,"x":0.5, "title" : "","itemwidth" : 60
                    },
                    
                    "showlegend": True

        }

for i in range (1,18):
    layout[f"xaxis{i}"] = { 'showline': True, "tickangle": -45,'showgrid': True,'gridcolor': 'lightgrey',
        'gridwidth': 1, 'linewidth': 2,  'linecolor': 'black',  'mirror': True}
    layout[f"yaxis{i}"] = {'zerolinewidth': 2,'zerolinecolor': 'black', 'showline': True,
                           'showgrid': True,'gridcolor': 'lightgrey','gridwidth': 1, 
                           'linewidth': 2,  'linecolor': 'black',  'mirror': True}

In [ ]:

figure = sp.make_subplots(rows = 4 , cols = 2, shared_xaxes = True, shared_yaxes = True,column_titles=["FR", "CZ"], 
                          row_titles=["SS", "SSOP" , "OPHR" , "SSOPHR" ],vertical_spacing=0.05, 
                          horizontal_spacing=0.055,  y_title= "Power [MW]")
                #  subplot_titles = ["SS [FR]" , "SS [CZ]", "SSOP [FR]" , "SSOP [CZ]", "OPHR [FR]" , "OPHR [CZ]","SSOPHR [FR]" , "SSOPHR [CZ]"])

import plotly.graph_objects as go

x = df_czech.index

colors = {"SS": "#70B0E0", "SSOP": "#D82C20", "OPHR": "#108372", "SSOPHR": "#F18F49", "SSOPHRF": "#B66DFF" , "ref" : "#F1C716"}
figure.add_traces([go.Scatter(y=df_france['Consumption_S1']/1000, x =x, mode='lines', name='SS', line={"color": colors['SS']}),
                    go.Scatter(y=df_czech['Consumption_S1']/1000,x =x, mode='lines', name='SS [CZ]', line={"color": colors['SS']},showlegend=False),
                   go.Scatter(y=df_france['Consumption_S2']/1000,x =x, mode='lines', name='SSOP', line={"color": colors['SSOP']}),
                     go.Scatter(y=df_czech['Consumption_S2']/1000, x =x, mode='lines', name='SSOP [CZ]', line={"color": colors['SSOP']},showlegend=False),
                   go.Scatter(y=df_france['Consumption_S3']/1000,x =x, mode='lines', name='OPHR', line={"color": colors['OPHR']}),
                     go.Scatter(y=df_czech['Consumption_S3']/1000,x =x, mode='lines', name='OPHR [CZ]', line={"color": colors['OPHR']},showlegend=False),
                   go.Scatter(y=df_france['Consumption_S4']/1000,x =x, mode='lines', name='SSOPHR', line={"color": colors['SSOPHR']}),
                     go.Scatter(y=df_czech['Consumption_S4']/1000,x =x, mode='lines', name='SSOPHR [CZ]', line={"color": colors['SSOPHR']},showlegend=False),
                   
                   ]
                  
                   , rows = [1,1,2,2,3,3,4,4] , cols= [1,2,1,2,1,2,1,2])
figure.update_layout(layout, title = {"text" : "Consumption Profile of LNCMI for Each Scenario" , "x" : 0.7, "y" : 0.99,
                                       "xanchor" : "center", "font" : {"color" : "black"}})
figure.layout.title['font'] = {"color" : "black"}
figure.layout.title['xanchor'] = "right"
for annot in figure.layout["annotations"]:
    annot["font"]["size"] = 32
    annot["font"]["color"] = "black"
    # annot['xanchor'] = "left"

figure

In [ ]:
for col in df_france.columns:
    if "CO2" and 'total' in col:
        print(col)
(1642.8286792383851 - (df_czech[[col for col in df_france.columns if "CO2" in col and "total" in col]].sum()/1e6))/1642.8286792383851

estimated_CO2_S1_total
estimated_CO2_S2_total
estimated_CO2_S3_total
estimated_CO2_S4_total


estimated_CO2_S1_total   -0.065737
estimated_CO2_S2_total    0.040800
estimated_CO2_S3_total   -0.185617
estimated_CO2_S4_total    0.175716
dtype: float64

In [ ]:

100*(1642.8286792383851 - (df_czech[[col for col in df_czech.columns if "CO2" in col and "total" in col]].sum()/1e6))/1642.8286792383851

estimated_CO2_S1_total    -6.573676
estimated_CO2_S2_total     4.080030
estimated_CO2_S3_total   -18.561660
estimated_CO2_S4_total    17.571588
dtype: float64

In [ ]:
df_czech[[col for col in df_france.columns if "S3" in col ]].sum()/1e3

grid_import_S3_kWh                 4.082386e+03
grid_export_S3_kWh                 0.000000e+00
OPEX_S3_€                          1.347187e+03
Buffer_Cap_S3_KWp                  0.000000e+00
Buffer_Power_S3_KW                 0.000000e+00
Buffer_Energy_S3_KW                0.000000e+00
heatpump_elec_Power_S3_KW          0.000000e+00
heatpump_thermal_Power_S3_KW       0.000000e+00
heatpump_elec_binary_S3            2.520000e+00
surplus_elec_cons_Power_S3_kW      8.312433e+01
surplus_therm_prod_Power_S3_kW     7.065568e+01
estimated_CO2_S3_elec              1.884726e+06
estimated_CO2_S3_district_heat     6.303922e+04
estimated_CO2_S3_buffer            0.000000e+00
estimated_CO2_S3_heatpump          0.000000e+00
estimated_CO2_S3_total             1.947765e+06
Consumption_S3                     3.999262e+03
Consumption_S3_compare             3.999262e+03
heat_demand_district_heat_S3_kW    6.927387e+02
heat_demand_district_heat_S3_MW    6.927387e-01
dtype: float64

In [ ]:
df_czech[[col for col in df_france.columns if "S1" in col ]].sum()/1e3

PV_Cap_S1_KWp             4.712397e+04
grid_import_S1_kWh        2.851822e+03
grid_export_S1_kWh        4.383473e+03
OPEX_S1_€                 4.852959e+02
estimated_CO2_S1_elec     1.475950e+06
estimated_CO2_S1_pv       2.118340e+05
estimated_CO2_S1_total    1.750823e+06
Consumption_S1            3.999262e+03
PV_production_S1          5.530914e+03
dtype: float64

In [ ]:
df_france["Heat_demand[kW]"] = df_france["Heat_demand[MW]"]*1000
df_france["Heat_demand[kW]"].gt(0.0001).sum()

1367

In [ ]:
len(df_france)

2520

In [ ]:
df_czech.sum()

Electricity_Consumption[MW]        3.999262e+03
Taux de Co2                        1.259595e+06
Heat_demand[MW]                    6.927387e+02
cciag_co2                          2.293200e+05
pv                                 2.957710e+02
                                       ...     
heat_demand_district_heat_S4_kW    3.286074e+05
heat_demand_district_heat_S4_MW    3.286074e+02
electric_emissions_ref             2.002937e+09
thermal_emissions_ref              6.303922e+07
Reference_emissions                2.065977e+09
Length: 73, dtype: float64

In [ ]:
ttt = df_czech[["Taux de Co2","Electricity_Consumption[MW]"]].product(axis=1) + df_czech[["Heat_demand[MW]" , "cciag_co2"]].product(axis =1)


ttt.sum()/1e3


2065.9766312817183

In [ ]:

figure = sp.make_subplots(rows = 5 , cols = 2, shared_xaxes = True, shared_yaxes = False,column_titles=["FR", "CZ"], 
                          row_titles=["Reference", "SS", "SSOP" , "OPHR" , "SSOPHR"],vertical_spacing=0.05, 
                          horizontal_spacing=0.055,  y_title= "Carbon Footprint [Tonnes CO₂eq]")
                #  subplot_titles = ["SS [FR]" , "SS [CZ]", "SSOP [FR]" , "SSOP [CZ]", "OPHR [FR]" , "OPHR [CZ]","SSOPHR [FR]" , "SSOPHR [CZ]"])

import plotly.graph_objects as go

x = df_czech.index
figure.add_traces([go.Scatter(y=df_france['Reference_emissions']/1e6, x =x, mode='lines', name='ref', line={"color":colors["ref"]}),
                    go.Scatter(y=df_czech['Reference_emissions']/1e6,x =x, mode='lines', name='ref" [CZ]', line={"color":colors["ref"]},showlegend=False),
                   go.Scatter(y=df_france['estimated_CO2_S1_total']/1e6, x =x, mode='lines', name='SS', line={"color":colors["SS"]}),
                    go.Scatter(y=df_czech['estimated_CO2_S1_total']/1e6,x =x, mode='lines', name='SS [CZ]', line={"color":colors["SS"]},showlegend=False),
                   go.Scatter(y=df_france['estimated_CO2_S2_total']/1e6,x =x, mode='lines', name='SSOP', line={"color":colors["SSOP"]}),
                     go.Scatter(y=df_czech['estimated_CO2_S2_total']/1e6, x =x, mode='lines', name='SSOP [CZ]', line={"color":colors["SSOP"]},showlegend=False),
                   go.Scatter(y=df_france['estimated_CO2_S3_total']/1e6,x =x, mode='lines', name='OPHR', line={"color":colors["OPHR"]}),
                     go.Scatter(y=df_czech['estimated_CO2_S3_total']/1e6,x =x, mode='lines', name='OPHR [CZ]', line={"color":colors["OPHR"]},showlegend=False),
                   go.Scatter(y=df_france['estimated_CO2_S4_total']/1e6,x =x, mode='lines', name='SSOPHR', line={"color":colors["SSOPHR"]}),
                     go.Scatter(y=df_czech['estimated_CO2_S4_total']/1e6,x =x, mode='lines', name='SSOPHR [CZ]', line={"color":colors["SSOPHR"]},showlegend=False),
                   
                   ]
                  
                   , rows = [1,1,2,2,3,3,4,4,5,5] , cols= [1,2,1,2,1,2,1,2,1,2])
figure.update_layout(layout,height = 2000, width = 2200,  title = {"text" : "Carbon Footprint of LNCMI for Each Scenario" , "x" : 0.7, "y" : 0.99,
                                       "xanchor" : "center", "font" : {"color" : "black"}})
figure.layout.title['font'] = {"color" : "black"}
figure.layout.title['xanchor'] = "right"
for annot in figure.layout["annotations"]:
    annot["font"]["size"] = 32
    annot["font"]["color"] = "black"
    # annot['xanchor'] = "left"

figure

In [ ]:

figure = sp.make_subplots(rows = 2 , cols = 1, shared_xaxes = True, shared_yaxes = True,vertical_spacing=0.05, 
                          horizontal_spacing=0.055,  y_title= "Carbon Intensity [gCO₂eq/kWh]")
                #  subplot_titles = ["SS [FR]" , "SS [CZ]", "SSOP [FR]" , "SSOP [CZ]", "OPHR [FR]" , "OPHR [CZ]","SSOPHR [FR]" , "SSOPHR [CZ]"])

import plotly.graph_objects as go

x = df_czech.index
figure.add_traces([
                    go.Scatter(y=df_france['Taux de Co2'],x =x, mode='lines', name='FR', line={"color":"#009292"}),
                    go.Scatter(y=df_czech['Taux de Co2'],x =x, mode='lines', name='CZ', line={"color":"#002050"}),
                    
                   ]
                  
                   , rows = [1,2] , cols= [1,1])
figure.update_layout(layout,margin=dict(l=180), width = 1800, title = {"text" : "Electricity Grid Carbon Intensity [gCO₂eq/kWh] " , "x" : 0.5, "y" : 0.99,
                                       "xanchor" : "center", "font" : {"color" : "black"}})

for annot in figure.layout["annotations"]:
    annot["font"]["size"] = 30
    annot["font"]["color"] = "black"
    annot['xanchor'] = "right"
    annot['x'] = -0.04
    
figure

In [ ]:
figure.layout["annotations"]

(layout.Annotation({
     'font': {'color': 'black', 'size': 30},
     'showarrow': False,
     'text': 'Carbon Intensity [gCO₂eq/kWh]',
     'textangle': -90,
     'x': -0.04,
     'xanchor': 'right',
     'xref': 'paper',
     'xshift': -40,
     'y': 0.5,
     'yanchor': 'middle',
     'yref': 'paper'
 }),)

In [ ]:
px.line(df_france[[col for col in df_france.columns if "Consumption_S" in col]])

In [ ]:
px.line(df_czech[[col for col in df_france.columns if "Consumption_S" in col]])

In [ ]:

figure = sp.make_subplots(rows = 2 , cols = 2, shared_xaxes = True, shared_yaxes = True,column_titles=["FR", "CZ"], 
                          row_titles=[ "OPHR" ,  "SSOPHR"],vertical_spacing=0.05, 
                          horizontal_spacing=0.055,  y_title= "Power [MW]")
                #  subplot_titles = ["SS [FR]" , "SS [CZ]", "SSOP [FR]" , "SSOP [CZ]", "OPHR [FR]" , "OPHR [CZ]","SSOPHR [FR]" , "SSOPHR [CZ]"])

import plotly.graph_objects as go

x = df_czech.index
figure.add_traces([
                    # go.Scatter(y=df_france['Buffer_Power_S3_KW']/1000, x =x, mode='lines', name='SS', line={"color":"blue"}),
                    # go.Scatter(y=df_czech['PV_production_S1']/1000,x =x, mode='lines', name='SS [CZ]', line={"color":"blue"},showlegend=False),
                    # go.Scatter(y=df_france['PV_production_S2']/1000,x =x, mode='lines', name='SSOP', line={"color":"red"}),
                    # go.Scatter(y=df_czech['PV_production_S2']/1000, x =x, mode='lines', name='SSOP [CZ]', line={"color":"red"},showlegend=False),
                    go.Scatter(y=df_france['Buffer_Power_S3_KW']/1000,x =x, mode='lines', name='OPHR', line={"color":colors["OPHR"]}),
                    go.Scatter(y=df_czech['Buffer_Power_S3_KW']/1000,x =x, mode='lines', name='OPHR [CZ]', line={"color":colors["OPHR"]},showlegend=False),
                    go.Scatter(y=df_france['Buffer_Power_S4_KW']/1000,x =x, mode='lines', name='SSOPHR', line={"color":colors["SSOPHR"]}),
                    go.Scatter(y=df_czech['Buffer_Power_S4_KW']/1000,x =x, mode='lines', name='SSOPHR [CZ]', line={"color":colors["SSOPHR"]},showlegend=False),
                    # go.Scatter(y=df_france['Buffer_Power_S5_KW']/1000,x =x, mode='lines', name='SSOPHRF', line={"color":colors["SSOPHRF"]}),
                    # go.Scatter(y=df_czech['Buffer_Power_S5_KW']/1000,x =x, mode='lines', name='SSOPHRF [CZ]', line={"color":colors["SSOPHRF"]},showlegend=False)
                   ]
                  
                   , rows = [1,1,2,2] , cols= [1,2,1,2])

layout["legend"]["y"] = -0.208

 


figure.update_layout(layout, title = {"text" : "Thermal Storage Charge/Discharge Scheduling" , "x" : 0.7, "y" : 0.99,
                                       "xanchor" : "center", "font" : {"color" : "black"}},height = 1000)
figure.layout.title['font'] = {"color" : "black"}
figure.layout.title['xanchor'] = "right"
for annot in figure.layout["annotations"]:
    annot["font"]["size"] = 32
    annot["font"]["color"] = "black"
    # annot['xanchor'] = "left"

figure

In [ ]:
for col in df_france.columns:
    if "test" in col:
        print(col)

df_czech[[col for col in df_czech.columns if "S3" in col]]

,grid_import_S3_kWh,grid_export_S3_kWh,OPEX_S3_€,Buffer_Cap_S3_KWp,Buffer_Power_S3_KW,Buffer_Energy_S3_KW,heatpump_elec_Power_S3_KW,heatpump_thermal_Power_S3_KW,heatpump_elec_binary_S3,surplus_elec_cons_Power_S3_kW,surplus_therm_prod_Power_S3_kW,estimated_CO2_S3_elec,estimated_CO2_S3_district_heat,estimated_CO2_S3_buffer,estimated_CO2_S3_heatpump,estimated_CO2_S3_total,Consumption_S3,Consumption_S3_compare,heat_demand_district_heat_S3_kW,heat_demand_district_heat_S3_MW
datetime,,,,,,,,,,,,,,,,,,,,
2023-01-24 00:00:00,45.283333,0.0,14.94350,0.0,0.0,0.0,0.0,0.0,1.0,1.950,1.65750,27341.623833,34580.455,0.0,0.0,61922.078833,43.333333,43.333333,380.005,0.380005
2023-01-24 01:00:00,45.283333,0.0,14.94350,0.0,0.0,0.0,0.0,0.0,1.0,1.950,1.65750,27513.700500,37317.735,0.0,0.0,64831.435500,43.333333,43.333333,410.085,0.410085
2023-01-24 02:00:00,43.541667,0.0,14.36875,0.0,0.0,0.0,0.0,0.0,1.0,1.875,1.59375,26283.056268,41203.435,0.0,0.0,67486.491268,41.666667,41.666667,452.785,0.452785
2023-01-24 03:00:00,45.283333,0.0,14.94350,0.0,0.0,0.0,0.0,0.0,1.0,1.950,1.65750,26921.847333,42580.720,0.0,0.0,69502.567333,43.333333,43.333333,467.920,0.467920
2023-01-24 04:00:00,43.541667,0.0,14.36875,0.0,0.0,0.0,0.0,0.0,1.0,1.875,1.59375,26094.085417,71746.675,0.0,0.0,97840.760417,41.666667,41.666667,788.425,0.788425
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-05-08 19:00:00,0.000000,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,1.0,0.000,0.00000,0.000000,0.000,0.0,0.0,0.000000,0.000000,0.000000,0.000,0.000000
2023-05-08 20:00:00,0.000000,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,1.0,0.000,0.00000,0.000000,0.000,0.0,0.0,0.000000,0.000000,0.000000,0.000,0.000000
2023-05-08 21:00:00,0.000000,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,1.0,0.000,0.00000,0.000000,0.000,0.0,0.0,0.000000,0.000000,0.000000,0.000,0.000000


In [ ]:

# figure = sp.make_subplots(rows = 2 , cols = 2, shared_xaxes = True, shared_yaxes = False,column_titles=["Power [MW]", "Energy [MWh]"], 
#                           row_titles=[ "GWP/kwh_dis" ,  "GWP"],vertical_spacing=0.05, 
#                           horizontal_spacing=0.055,  y_title= "Power [MW]")
#                 #  subplot_titles = ["SS [FR]" , "SS [CZ]", "SSOP [FR]" , "SSOP [CZ]", "OPHR [FR]" , "OPHR [CZ]","SSOPHR [FR]" , "SSOPHR [CZ]"])

# import plotly.graph_objects as go

# x = df_czech.index
# figure.add_traces([
#                     # go.Scatter(y=df_france['Buffer_Power_S3_KW']/1000, x =x, mode='lines', name='SS', line={"color":"blue"}),
#                     # go.Scatter(y=df_czech['PV_production_S1']/1000,x =x, mode='lines', name='SS [CZ]', line={"color":"blue"},showlegend=False),
#                     # go.Scatter(y=df_france['PV_production_S2']/1000,x =x, mode='lines', name='SSOP', line={"color":"red"}),
#                     # go.Scatter(y=df_czech['PV_production_S2']/1000, x =x, mode='lines', name='SSOP [CZ]', line={"color":"red"},showlegend=False),
#                     go.Scatter(y=df_france['Buffer_Power_S3_KW']/1000,x =x, mode='lines', name='OPHR', line={"color":"green"}),
#                     go.Scatter(y=df_france['Buffer_Energy_S3_KW']/1000,x =x, mode='lines', name='OPHR [CZ]', line={"color":"green"},showlegend=False),
#                     go.Scatter(y=df_france['Buffer_Power_S3_KW_test']/1000,x =x, mode='lines', name='SSOPHR', line={"color":"orange"}),
#                     go.Scatter(y=df_france['Buffer_Energy_S3_KW_test']/1000,x =x, mode='lines', name='SSOPHR [CZ]', line={"color":"orange"},showlegend=False)
#                    ]
                  
#                    , rows = [1,1,2,2] , cols= [1,2,1,2])
# figure.update_layout(layout, title = {"text" : "Thermal Storage Charge/Discharge Scheduling for Scenario 3 [FRANCE]" , "x" : 0.7, "y" : 0.99,
#                                        "xanchor" : "center", "font" : {"color" : "black"}})
# figure.layout.title['font'] = {"color" : "black"}
# figure.layout.title['xanchor'] = "right"
# for annot in figure.layout["annotations"]:
#     annot["font"]["size"] = 32
#     annot["font"]["color"] = "black"
#     # annot['xanchor'] = "left"

# figure

In [ ]:

# figure = sp.make_subplots(rows = 2 , cols = 2, shared_xaxes = True, shared_yaxes = False,column_titles=["Power [MW]", "Energy [MWh]"], 
#                           row_titles=[ "GWP/kwh_dis" ,  "GWP"],vertical_spacing=0.05, 
#                           horizontal_spacing=0.055,  y_title= "Power [MW]")
#                 #  subplot_titles = ["SS [FR]" , "SS [CZ]", "SSOP [FR]" , "SSOP [CZ]", "OPHR [FR]" , "OPHR [CZ]","SSOPHR [FR]" , "SSOPHR [CZ]"])

# import plotly.graph_objects as go

# x = df_czech.index
# figure.add_traces([
#                     # go.Scatter(y=df_france['Buffer_Power_S3_KW']/1000, x =x, mode='lines', name='SS', line={"color":"blue"}),
#                     # go.Scatter(y=df_czech['PV_production_S1']/1000,x =x, mode='lines', name='SS [CZ]', line={"color":"blue"},showlegend=False),
#                     # go.Scatter(y=df_france['PV_production_S2']/1000,x =x, mode='lines', name='SSOP', line={"color":"red"}),
#                     # go.Scatter(y=df_czech['PV_production_S2']/1000, x =x, mode='lines', name='SSOP [CZ]', line={"color":"red"},showlegend=False),
#                     go.Scatter(y=df_france['Buffer_Power_S4_KW']/1000,x =x, mode='lines', name='OPHR', line={"color":"green"}),
#                     go.Scatter(y=df_france['Buffer_Energy_S4_KW']/1000,x =x, mode='lines', name='OPHR [CZ]', line={"color":"green"},showlegend=False),
#                     go.Scatter(y=df_france['Buffer_Power_S4_KW_test']/1000,x =x, mode='lines', name='SSOPHR', line={"color":"orange"}),
#                     go.Scatter(y=df_france['Buffer_Energy_S4_KW_test']/1000,x =x, mode='lines', name='SSOPHR [CZ]', line={"color":"orange"},showlegend=False)
#                    ]
                  
#                    , rows = [1,1,2,2] , cols= [1,2,1,2])
# figure.update_layout(layout, title = {"text" : "Thermal Storage Charge/Discharge Scheduling for Scenario 4 [FRANCE]" , "x" : 0.7, "y" : 0.99,
#                                        "xanchor" : "center", "font" : {"color" : "black"}})
# figure.layout.title['font'] = {"color" : "black"}
# figure.layout.title['xanchor'] = "right"
# for annot in figure.layout["annotations"]:
#     annot["font"]["size"] = 32
#     annot["font"]["color"] = "black"
#     # annot['xanchor'] = "left"

# figure

In [ ]:

# figure = sp.make_subplots(rows = 2 , cols = 2, shared_xaxes = True, shared_yaxes = False,column_titles=["Power [MW]", "Energy [MWh]"], 
#                           row_titles=[ "GWP/kwh_dis" ,  "GWP"],vertical_spacing=0.05, 
#                           horizontal_spacing=0.055,  y_title= "Power [MW]")
#                 #  subplot_titles = ["SS [FR]" , "SS [CZ]", "SSOP [FR]" , "SSOP [CZ]", "OPHR [FR]" , "OPHR [CZ]","SSOPHR [FR]" , "SSOPHR [CZ]"])

# import plotly.graph_objects as go

# x = df_czech.index
# figure.add_traces([
#                     # go.Scatter(y=df_czech['Buffer_Power_S3_KW']/1000, x =x, mode='lines', name='SS', line={"color":"blue"}),
#                     # go.Scatter(y=df_czech['PV_production_S1']/1000,x =x, mode='lines', name='SS [CZ]', line={"color":"blue"},showlegend=False),
#                     # go.Scatter(y=df_czech['PV_production_S2']/1000,x =x, mode='lines', name='SSOP', line={"color":"red"}),
#                     # go.Scatter(y=df_czech['PV_production_S2']/1000, x =x, mode='lines', name='SSOP [CZ]', line={"color":"red"},showlegend=False),
#                     go.Scatter(y=df_czech['Buffer_Power_S3_KW']/1000,x =x, mode='lines', name='OPHR', line={"color":"green"}),
#                     go.Scatter(y=df_czech['Buffer_Energy_S3_KW']/1000,x =x, mode='lines', name='OPHR [CZ]', line={"color":"green"},showlegend=False),
#                     go.Scatter(y=df_czech['Buffer_Power_S3_KW_test']/1000,x =x, mode='lines', name='SSOPHR', line={"color":"orange"}),
#                     go.Scatter(y=df_czech['Buffer_Energy_S3_KW_test']/1000,x =x, mode='lines', name='SSOPHR [CZ]', line={"color":"orange"},showlegend=False)
#                    ]
                  
#                    , rows = [1,1,2,2] , cols= [1,2,1,2])
# figure.update_layout(layout, title = {"text" : "Thermal Storage Charge/Discharge Scheduling for Scenario 4 [CZECH]" , "x" : 0.7, "y" : 0.99,
#                                        "xanchor" : "center", "font" : {"color" : "black"}})
# figure.layout.title['font'] = {"color" : "black"}
# figure.layout.title['xanchor'] = "right"
# for annot in figure.layout["annotations"]:
#     annot["font"]["size"] = 32
#     annot["font"]["color"] = "black"
#     # annot['xanchor'] = "left"

# figure

In [ ]:

# figure = sp.make_subplots(rows = 2 , cols = 2, shared_xaxes = True, shared_yaxes = False,column_titles=["Power [MW]", "Energy [MWh]"], 
#                           row_titles=[ "GWP/kwh_dis" ,  "GWP"],vertical_spacing=0.05, 
#                           horizontal_spacing=0.055,  y_title= "Power [MW]")
#                 #  subplot_titles = ["SS [FR]" , "SS [CZ]", "SSOP [FR]" , "SSOP [CZ]", "OPHR [FR]" , "OPHR [CZ]","SSOPHR [FR]" , "SSOPHR [CZ]"])

# import plotly.graph_objects as go

# x = df_czech.index
# figure.add_traces([
#                     # go.Scatter(y=df_czech['Buffer_Power_S3_KW']/1000, x =x, mode='lines', name='SS', line={"color":"blue"}),
#                     # go.Scatter(y=df_czech['PV_production_S1']/1000,x =x, mode='lines', name='SS [CZ]', line={"color":"blue"},showlegend=False),
#                     # go.Scatter(y=df_czech['PV_production_S2']/1000,x =x, mode='lines', name='SSOP', line={"color":"red"}),
#                     # go.Scatter(y=df_czech['PV_production_S2']/1000, x =x, mode='lines', name='SSOP [CZ]', line={"color":"red"},showlegend=False),
#                     go.Scatter(y=df_czech['Buffer_Power_S4_KW']/1000,x =x, mode='lines', name='OPHR', line={"color":"green"}),
#                     go.Scatter(y=df_czech['Buffer_Energy_S4_KW']/1000,x =x, mode='lines', name='OPHR [CZ]', line={"color":"green"},showlegend=False),
#                     go.Scatter(y=df_czech['Buffer_Power_S4_KW_test']/1000,x =x, mode='lines', name='SSOPHR', line={"color":"orange"}),
#                     go.Scatter(y=df_czech['Buffer_Energy_S4_KW_test']/1000,x =x, mode='lines', name='SSOPHR [CZ]', line={"color":"orange"},showlegend=False)
#                    ]
                  
#                    , rows = [1,1,2,2] , cols= [1,2,1,2])
# figure.update_layout(layout, title = {"text" : "Thermal Storage Charge/Discharge Scheduling for Scenario 4 [CZECH]" , "x" : 0.7, "y" : 0.99,
#                                        "xanchor" : "center", "font" : {"color" : "black"}})
# figure.layout.title['font'] = {"color" : "black"}
# figure.layout.title['xanchor'] = "right"
# for annot in figure.layout["annotations"]:
#     annot["font"]["size"] = 32
#     annot["font"]["color"] = "black"
#     # annot['xanchor'] = "left"

# figure

In [ ]:

figure = sp.make_subplots(rows = 2 , cols = 2, shared_xaxes = True, shared_yaxes = False,column_titles=["FR", "CZ"], 
                          row_titles=[ "OPHR" ,  "SSOPHR"],vertical_spacing=0.05, 
                          horizontal_spacing=0.055,  y_title= "Energy [MWh]")
                #  subplot_titles = ["SS [FR]" , "SS [CZ]", "SSOP [FR]" , "SSOP [CZ]", "OPHR [FR]" , "OPHR [CZ]","SSOPHR [FR]" , "SSOPHR [CZ]"])

import plotly.graph_objects as go

x = df_france.index
figure.add_traces([
                    # go.Scatter(y=df_france['Buffer_Energy_S3_KW']/1000, x =x, mode='lines', name='SS', line={"color":"blue"}),
                    # go.Scatter(y=df_france['PV_production_S1']/1000,x =x, mode='lines', name='SS [CZ]', line={"color":"blue"},showlegend=False),
                    # go.Scatter(y=df_france['PV_production_S2']/1000,x =x, mode='lines', name='SSOP', line={"color":"red"}),
                    # go.Scatter(y=df_france['PV_production_S2']/1000, x =x, mode='lines', name='SSOP [CZ]', line={"color":"red"},showlegend=False),
                    go.Scatter(y=df_france['Buffer_Energy_S3_KW']/1000,x =x, mode='lines', name='OPHR', line={"color":colors["OPHR"]}),
                    go.Scatter(y=df_czech['Buffer_Energy_S3_KW']/1000,x =x, mode='lines', name='FR [CZ]', line={"color":colors["OPHR"]},showlegend=False),

                    go.Scatter(y=df_france['Buffer_Energy_S4_KW']/1000,x =x, mode='lines', name='SSOPHR', line={"color":colors["SSOPHR"]}),
                    go.Scatter(y=df_czech['Buffer_Energy_S4_KW']/1000,x =x, mode='lines', name='CZ [CZ]', line={"color":colors["SSOPHR"]},showlegend=False),

                    
                   ]
                  
                   , rows = [1,1,2,2] , cols= [1,2,1,2])
figure.update_layout(layout, height = 1000, 
                    #  title = {"text" : "Evolution of Energy stored in Dimensioned Thermal Storage" , "x" : 0.7, "y" : 0.99,
                    #                    "xanchor" : "center", "font" : {"color" : "black"}},
                                        yaxis1 = {'zerolinewidth': 0,  'zerolinecolor':'grey'},yaxis3 = {'zerolinewidth': 0,  'zerolinecolor':'grey'},
                                        yaxis2 = {'zerolinewidth': 0,  'zerolinecolor':'grey'},yaxis4 = {'zerolinewidth': 0,  'zerolinecolor':'grey'}
                                      
                                       )
figure.layout.title['font'] = {"color" : "black"}
figure.layout.title['xanchor'] = "right"
for annot in figure.layout["annotations"]:
    annot["font"]["size"] = 30
    annot["font"]["color"] = "black"
    # annot['xanchor'] = "left"

figure

In [ ]:

figure = sp.make_subplots(rows = 3 , cols = 2, shared_xaxes = True, shared_yaxes = False,column_titles=["FR", "CZ"], 
                          row_titles=["SS", "SSOP" ,  "SSOPHR"],vertical_spacing=0.05, 
                          horizontal_spacing=0.055,  y_title= "Power [MW]")
                #  subplot_titles = ["SS [FR]" , "SS [CZ]", "SSOP [FR]" , "SSOP [CZ]", "OPHR [FR]" , "OPHR [CZ]","SSOPHR [FR]" , "SSOPHR [CZ]"])

import plotly.graph_objects as go

x = df_czech.index
figure.add_traces([go.Scatter(y=df_france['PV_production_S1']/1000, x =x, mode='lines', name='SS', line={"color":"blue"}),
                    go.Scatter(y=df_czech['PV_production_S1']/1000,x =x, mode='lines', name='SS [CZ]', line={"color":"blue"},showlegend=False),
                   go.Scatter(y=df_france['PV_production_S2']/1000,x =x, mode='lines', name='SSOP', line={"color":"red"}),
                     go.Scatter(y=df_czech['PV_production_S2']/1000, x =x, mode='lines', name='SSOP [CZ]', line={"color":"red"},showlegend=False),
                   go.Scatter(y=df_france['PV_production_S4']/1000,x =x, mode='lines', name='SSOPHR', line={"color":"orange"}),
                     go.Scatter(y=df_czech['PV_production_S4']/1000,x =x, mode='lines', name='SSOPHR [CZ]', line={"color":"orange"},showlegend=False),
                #    go.Scatter(y=df_france['Consumption_S4']/1000,x =x, mode='lines', name='SSOPHR', line={"color":"orange"}),
                #      go.Scatter(y=df_czech['Consumption_S4']/1000,x =x, mode='lines', name='SSOPHR [CZ]', line={"color":"orange"},showlegend=False)
                   ]
                  
                   , rows = [1,1,2,2,3,3] , cols= [1,2,1,2,1,2])
figure.update_layout(layout, title = {"text" : "PV production Profiles" , "x" : 0.7, "y" : 0.99,
                                       "xanchor" : "center", "font" : {"color" : "black"}})
figure.layout.title['font'] = {"color" : "black"}
figure.layout.title['xanchor'] = "right"
for annot in figure.layout["annotations"]:
    annot["font"]["size"] = 32
    annot["font"]["color"] = "black"
    # annot['xanchor'] = "left"

figure

In [ ]:
for col in  df_france.columns:
    if "istrict" in col.lower():
        print (col)

district_heating_price[€/kWh]
estimated_CO2_district_heating
estimated_CO2_S3_district_heat
heat_demand_district_heat_S3_kW
heat_demand_district_heat_S3_MW
estimated_CO2_S4_district_heat
heat_demand_district_heat_S4_kW
heat_demand_district_heat_S4_MW


In [ ]:
df_france.max()

Electricity_Consumption[MW]        1.978683e+01
Taux de Co2                        1.010000e+02
Heat_demand[MW]                    1.712290e+00
cciag_co2                          9.100000e+01
pv                                 7.517495e-01
                                       ...     
heat_demand_district_heat_S4_MW    0.000000e+00
electric_emissions_ref             1.550565e+06
thermal_emissions_ref              1.558184e+05
Reference_emissions                1.550565e+06
Heat_demand[kW]                    1.712290e+03
Length: 74, dtype: float64

In [ ]:
layout = {"height" : 1800, "width": 2000, "template" : "plotly_white",
        "font":{"size" : 22},

        "legend" : {"tracegroupgap":8 ,"font_size": 28, "orientation":"h", "yanchor":"bottom","xanchor":"center",
                    "y":-0.12,"x":0.5, "title" : "","itemwidth" : 60
                    },
                    
                    "showlegend": True

        }

for i in range (1,18):
    layout[f"xaxis{i}"] = { 'showline': True, "tickangle": -45,'showgrid': True,'gridcolor': 'lightgrey',
        'gridwidth': 1, 'linewidth': 2,  'linecolor': 'black',  'mirror': True}
    layout[f"yaxis{i}"] = {'zerolinewidth': 1,'zerolinecolor': 'black', 'showline': True,
                           'showgrid': True,'gridcolor': 'lightgrey','gridwidth': 1, 
                           'linewidth': 2,  'linecolor': 'black',  'mirror': True}

In [ ]:

figure = sp.make_subplots(rows = 2 , cols = 2, shared_xaxes = True, shared_yaxes = True,column_titles=["FR", "CZ"], 
                          row_titles=[ "OPHR" ,  "SSOPHR"],vertical_spacing=0.05, 
                          horizontal_spacing=0.055,  y_title= "Power [MW]")
                #  subplot_titles = ["SS [FR]" , "SS [CZ]", "SSOP [FR]" , "SSOP [CZ]", "OPHR [FR]" , "OPHR [CZ]","SSOPHR [FR]" , "SSOPHR [CZ]"])

import plotly.graph_objects as go

x = df_france.index
figure.add_traces([
                    # go.Scatter(y=df_france['Buffer_Energy_S3_KW']/1000, x =x, mode='lines', name='SS', line={"color":"blue"}),
                    # go.Scatter(y=df_france['PV_production_S1']/1000,x =x, mode='lines', name='SS [CZ]', line={"color":"blue"},showlegend=False),
                    # go.Scatter(y=df_france['PV_production_S2']/1000,x =x, mode='lines', name='SSOP', line={"color":"red"}),
                    # go.Scatter(y=df_france['PV_production_S2']/1000, x =x, mode='lines', name='SSOP [CZ]', line={"color":"red"},showlegend=False),
                    go.Scatter(y=df_france['heatpump_thermal_Power_S3_KW']/1000,x =x, mode='lines', name='Heat pump', line={"color":"#212ADE","width": 3},legendgroup='hp' ),
                    go.Scatter(y=df_france['heat_demand_district_heat_S3_MW'],x =x, mode='lines', name='District Heat', line={"color":"#C63D2B","dash":'dashdot',"width": 3},legendgroup='dh'),
                    go.Scatter(y=df_france['Heat_demand[MW]'],x =x, name='CNRS Heat Demand', line={"color":"#3CC919","width": 2,"dash":'longdash'},legendgroup='cn'),
                    
                    go.Scatter(y=df_czech['heatpump_thermal_Power_S3_KW']/1000,x =x, mode='lines', name='Heat pump [CZ]', line={"color":"#212ADE","width": 3},showlegend=False,legendgroup='hp'),
                    go.Scatter(y=df_czech['heat_demand_district_heat_S3_MW'],x =x, mode='lines', name='District Heat[CZ]', line={"color":"#C63D2B","dash":'dashdot',"width": 3},showlegend=False,legendgroup='dh'),
                    go.Scatter(y=df_czech['Heat_demand[MW]'],x =x, mode='lines', name='CNRS Heat Demand[CZ]', line={"color":"#3CC919","dash":'longdash',"width": 2},showlegend=False,legendgroup='cn'),
                    
                   go.Scatter(y=df_france['heatpump_thermal_Power_S4_KW']/1000,x =x, mode='lines', name='Heat pump', line={"color":"#212ADE","width": 3},showlegend=False,legendgroup='hp'),
                    go.Scatter(y=df_france['heat_demand_district_heat_S4_MW'],x =x, mode='lines', name='District Heat', line={"color":"#C63D2B","dash":'dashdot',"width": 3},showlegend=False,legendgroup='dh'),
                    go.Scatter(y=df_france['Heat_demand[MW]'],x =x, mode='lines', name='CNRS Heat Demand', line={"color":"#3CC919","dash":'longdash',"width": 2},showlegend=False,legendgroup='cn'),
                    
                    go.Scatter(y=df_czech['heatpump_thermal_Power_S4_KW']/1000,x =x, mode='lines', name='Heat pump [CZ]', line={"color":"#212ADE","width": 3},showlegend=False,legendgroup='hp'),
                    go.Scatter(y=df_czech['heat_demand_district_heat_S4_MW'],x =x, mode='lines', name='District Heat[CZ]', line={"color":"#C63D2B","dash":'dashdot',"width": 3},showlegend=False,legendgroup='dh'),
                    go.Scatter(y=df_czech['Heat_demand[MW]'],x =x, mode='lines', name='CNRS Heat Demand[CZ]', line={"color":"#3CC919","dash":'longdash',"width": 2},showlegend=False,legendgroup='cn')
                    
                   ]
                  
                   , rows = [1,1,1,1,1,1,2,2,2,2,2,2] , cols= [1,1,1,2,2,2,1,1,1,2,2,2,])
figure.update_layout(layout, height = 1500, width = 2800,
                    #  title = {"text" : "Heat Pump Vs District Heating Vs CNRS Campus Demand" , "x" : 0.7, "y" : 0.99,
                    #                    "xanchor" : "center", "font" : {"color" : "black"}}
                    )
figure.layout.title['font'] = {"color" : "black"}
figure.layout.title['xanchor'] = "right"
for annot in figure.layout["annotations"]:
    annot["font"]["size"] = 32
    annot["font"]["color"] = "black"
    # annot['xanchor'] = "left"

figure